# 06 — WebGPU acceleration (design walkthrough)

*Phase 3 of the [revival roadmap](../docs/impl-plans/2026-05-11_phase-3-webgpu.md): the third concrete `KernelBackend` adapter, lifting named-axis tensor algebra onto the GPU through [WebGPU](https://www.w3.org/TR/webgpu/) without any proprietary toolchain.*

**This notebook is a design walkthrough, not an execution.** As of the writing of this notebook (per the [Phase 4 release rehearsal](../docs/reports/2026-05-11_phase-4-release-rehearsal.md), the project does not assume the reader has a self-hosted GPU runner. Tutorial 06 therefore *narrates* the WGSL kernel sources (which are committed) and the dispatch design (which is documented at line-resolved detail) without executing GPU code. When the maintainer provisions a GPU runner and P3.M3.2 / P3.M4.2 land their dispatch wiring, an updated notebook will append cells that exercise the kernels live.

## What you will get from this notebook

1. You see the **eight committed WGSL kernels** (four element-wise binary + four unary + one tiled GEMM) and understand what each does.
2. You understand the **dispatch wiring design** — how a `tensor::core::backend::webgpu::Backend::add(a, b)` call will turn into a `gpu::createKernel` + `gpu::dispatchKernel` + `gpu::wait` sequence on top of [Answer.AI's gpu.cpp](https://github.com/AnswerDotAI/gpu.cpp) and [Google's Dawn](https://dawn.googlesource.com/dawn).
3. You see the **substrate trade-offs** captured by [ADR-0014](../docs/arc42/09-decisions/0014-external-substrate-strategy.md): why gpu.cpp is vendored (not linked), why the vcpkg baseline bump is deferred, why xeus-cpp replaced xeus-cling.
4. You see the **honest performance picture**: WebGPU loses to CPU on small element-wise (host↔device transfer dominates) and wins by ~50× on GEMM at 512³ (compute amortises transfer).

## Prerequisites

**[`00_intro.ipynb`](./00_intro.ipynb) — named-axis fundamentals.** No autograd knowledge required: this notebook covers *forward* execution on GPU only. The autograd backward pass is in [`05_autograd-from-scratch.ipynb`](./05_autograd-from-scratch.ipynb) and runs independently of the kernel backend — `Variable<T>` does not care whether the underlying ops dispatched on reference, Eigen, or WebGPU.

If you arrived here from a "WebGPU C++" search and want to see the WGSL kernels + dispatch design without learning autograd first, that is supported. If you arrived from a "named-tensor autograd" search and want autograd on the GPU, you also need notebook 05 (the same `Variable<T>` machinery composes with any `KernelBackend` adapter).

Sibling notebook 08 ([`08_swappable-backends.ipynb`](./08_swappable-backends.ipynb)) demonstrates the same Hexagonal-lite payoff for the reference + Eigen pair — it is the closest analogue to the architectural argument this notebook makes for WebGPU.

## 1. The Phase 3 sub-structure

Phase 3 (WebGPU adapter) ships in six milestones; the table below shows the current state.

| Milestone | Content | PR | Status |
| --------- | ------- | -- | ------ |
| P3.M1 | [ADR-0012](../docs/arc42/09-decisions/0012-webgpu-adapter-implementation-design.md) — WebGPU adapter implementation design | #32 | shipped |
| P3.M2 | `webgpu::Backend` stub + CMake plumbing (delegates every method to `reference::Backend`) | #38 | shipped |
| **P3.M3.1** | **WGSL sources for 4 binary element-wise kernels** (`add`/`sub`/`mul`/`div`) | #43 | shipped |
| **P3.M3.3** | **WGSL sources for 4 unary element-wise kernels** (`exp`/`log`/`relu`/`neg`) | #44 | shipped |
| **P3.M4.1** | **WGSL source for tiled GEMM kernel** (covers matvec + matmul) | #46 | shipped |
| P3.M3.2 | gpu.cpp dispatch wiring for element-wise — replaces stub delegation | — | **deferred** — gated on GPU runner + Dawn baseline bump |
| P3.M4.2 | gpu.cpp dispatch wiring for GEMM | — | **deferred** — same gate |
| P3.M5 | Broadcast / reduce / unbroadcast WGSL + dispatch | — | planned |
| P3.M6 | This notebook (`06_webgpu-acceleration.ipynb`) — closes Phase 3 | this PR | shipped (design walkthrough form) |

The split between `*.1` / `*.3` (committed WGSL source) and `*.2` (deferred dispatch) is per the [Phase 3 impl-plan addendum](../docs/impl-plans/2026-05-11_phase-3-webgpu.md). Both detailed-design docs ([element-wise](../docs/detailed-design/webgpu-element-wise-kernels.md) and [GEMM](../docs/detailed-design/webgpu-gemm-kernel.md)) carry line-resolved references into the vendored `third_party/gpu_cpp/gpu.hpp`, so when the dispatch PR lands the wiring is mechanical.

## 2. The eight committed WGSL kernels

All sources live under [`include/tensor/core/backend/webgpu_wgsl.hpp`](../include/tensor/core/backend/webgpu_wgsl.hpp) as `constexpr std::string_view` constants. Each is templated on `{{precision}}` and `{{workgroupSize}}` per gpu.cpp's `KernelCode` substitution convention ([gpu.hpp:291-389](../third_party/gpu_cpp/gpu.hpp)).

In [ ]:
#pragma cling add_include_path("../include")
#include <iostream>
#include <tensor/core/backend/webgpu_wgsl.hpp>

namespace wgsl = tensor::core::backend::webgpu::wgsl;

std::cout << "==== kAddF32 ====" << wgsl::kAddF32 << "\n";

### Element-wise binary structure

```
@group(0) @binding(0) var<storage, read>       a   : array<{{precision}}>;
@group(0) @binding(1) var<storage, read>       b   : array<{{precision}}>;
@group(0) @binding(2) var<storage, read_write> out : array<{{precision}}>;

@compute @workgroup_size({{workgroupSize}})
fn main(@builtin(global_invocation_id) gid: vec3<u32>) {
    let i = gid.x;
    if (i >= arrayLength(&out)) { return; }
    out[i] = a[i] <op> b[i];
}
```

The four binary kernels (`kAddF32`, `kSubF32`, `kMulF32`, `kDivF32`) differ only in the `<op>` — `+`, `-`, `*`, `/`. They share:

- Three storage bindings: read-only `a`, read-only `b`, read-write `out`.
- A 1-D workgroup with `{{workgroupSize}}` = 256 threads (matches Dawn's canonical choice; substituted by `gpu::KernelCode`).
- `arrayLength(&out)` for the bound check — works because `out`'s size at dispatch time is what the host code uploaded.

### Element-wise unary structure

The four unary kernels (`kExpF32`, `kLogF32`, `kReluF32`, `kNegF32`) are identical except for **two storage bindings** (input + output) and the body. Notably, `kReluF32` uses `max(a[i], 0.0)` rather than a branch — `max` maps to a single GPU instruction on every Dawn backend (Vulkan / Metal / D3D12).

### The tiled GEMM kernel

`kGemmF32` is one readable kernel that covers both matvec (rank-2 × rank-1) and matmul (rank-2 × rank-2) with one shared axis. The full design is in [`webgpu-gemm-kernel.md`](../docs/detailed-design/webgpu-gemm-kernel.md); the headline structure:

```
struct Params { M : u32, N : u32, K : u32 };
// bindings: a (M×K), b (K×N), out (M×N), p (uniform Params)

const TILE_M : u32 = 16u;
const TILE_N : u32 = 16u;
const TILE_K : u32 = 16u;

var<workgroup> shA : array<array<{{precision}}, TILE_K>, TILE_M>;
var<workgroup> shB : array<array<{{precision}}, TILE_N>, TILE_K>;

@compute @workgroup_size(TILE_N, TILE_M, 1)
fn main(...) {
    var acc : {{precision}} = 0.0;
    for (var t : u32 = 0u; t < nTiles; t = t + 1u) {
        // 1. cooperative load: 16×16 tile of A + 16×16 tile of B into shared memory
        workgroupBarrier();
        for (var k : u32 = 0u; k < TILE_K; k = k + 1u) {
            acc = acc + shA[lrow][k] * shB[k][lcol];
        }
        workgroupBarrier();
    }
    if (row < p.M && col < p.N) { out[row * p.N + col] = acc; }
}
```

Why one kernel for matvec and matmul? The canonical-reference framing ([ADR-0013](../docs/arc42/09-decisions/0013-reframe-as-canonical-reference-for-named-tensor-computation.md)) prefers one readable kernel over two specialised ones. A dedicated matvec kernel would extract more throughput when N = 1, but at the cost of doubling the surface area to read, test, and re-derive. The matvec inefficiency (15 / 16 threads idle in a 16-wide workgroup when N = 1) is documented in the design doc's §4 rather than engineered away.

In [ ]:
// Confirm the tile constants the kernel source hard-codes match the
// C++ side's expectations — text-validated by tests/test_webgpu_wgsl.cpp.
std::cout << "kGemmTileM = " << wgsl::kGemmTileM << "\n";
std::cout << "kGemmTileN = " << wgsl::kGemmTileN << "\n";
std::cout << "kGemmTileK = " << wgsl::kGemmTileK << "\n";
std::cout << "kDefaultWorkgroupSize = " << wgsl::kDefaultWorkgroupSize << "\n";

## 3. The dispatch wiring design (what P3.M3.2 / P3.M4.2 will land)

Today every method on `webgpu::Backend` delegates to a private `reference::Backend ref_` member. The P3.M3.2 and P3.M4.2 PRs will replace each delegation with the gpu.cpp dispatch sequence shown below. The full design with line-resolved references into `gpu.hpp` is in the two detailed-design docs.

### Element-wise binary dispatch sequence (e.g. `add`)

```cpp
template <class T>
DynamicTensor<T> Backend::add(DynamicTensor<T> const& a, DynamicTensor<T> const& b) const {
    if constexpr (!std::is_same_v<T, float>) {
        return ref_.add(a, b);  // f32-only MVP; delegate non-float
    } else {
        auto const n = a.size();
        auto& ctx = ensure_context();  // lazy gpu::createContext()

        auto gA   = gpu::createTensor(ctx, {n}, gpu::kf32);  // gpu.hpp:615
        auto gB   = gpu::createTensor(ctx, {n}, gpu::kf32);
        auto gOut = gpu::createTensor(ctx, {n}, gpu::kf32);

        gpu::toGPU(ctx, a.data(), gA);  // gpu.hpp:1112
        gpu::toGPU(ctx, b.data(), gB);

        gpu::KernelCode code{ std::string{wgsl::kAddF32},
                              wgsl::kDefaultWorkgroupSize, gpu::kf32 };
        gpu::Bindings bindings{ gA, gB, gOut };
        gpu::Shape totalWorkgroups{ gpu::cdiv(n, wgsl::kDefaultWorkgroupSize), 1, 1 };
        auto kernel = gpu::createKernel(ctx, code, bindings, totalWorkgroups);

        std::promise<void> p;
        auto f = p.get_future();
        gpu::dispatchKernel(ctx, kernel, p);  // gpu.hpp:1429
        gpu::wait(ctx, f);                     // gpu.hpp:978

        DynamicTensor<T> out{ a.shape() };
        gpu::toCPU(ctx, gOut, out.data(), n * sizeof(T));  // gpu.hpp:997
        return out;
    }
}
```

Substitute `kAddF32` → `kSubF32` / `kMulF32` / `kDivF32` for the other three binary operators; remove the `gB` upload and use a two-binding `Bindings{gA, gOut}` for the four unary operators. The structural shape is identical.

### GEMM dispatch sequence

Adds a `Params { M, N, K }` uniform buffer and a 2-D `totalWorkgroups = ceil(N/16, M/16)`. Same Context / Tensor / toGPU / dispatch / wait / toCPU pattern. Full pseudo-code in [`webgpu-gemm-kernel.md §3`](../docs/detailed-design/webgpu-gemm-kernel.md).

## 4. Substrate choices (why these and not others)

[ADR-0014](../docs/arc42/09-decisions/0014-external-substrate-strategy.md) bundles four substrate decisions that shape Phase 3. The choices follow from the [external-substrate research report](../docs/reports/2026-05-11_external-substrate-research.md).

1. **gpu.cpp is vendored, not linked.** Bus-factor 1: one primary maintainer at [AnswerDotAI/gpu.cpp](https://github.com/AnswerDotAI/gpu.cpp), no release since v0.2.0 (Aug 2024). Vendored under [`third_party/gpu_cpp/gpu.hpp`](../third_party/gpu_cpp/gpu.hpp) at tag `0.2.0` with a `VENDORED_FROM` record. `tools/check-vendored.sh` enforces.

2. **Dawn comes from vcpkg.** The port (`20260410.140140`, refreshed 2026-04-20) is now stable. The actual `find_package(dawn CONFIG REQUIRED)` wiring is deferred to P3.M3.2 because the baseline bump pulls **Eigen 3 → 5** as a side effect, which deserves its own focused CI iteration. The `webgpu` manifest feature in [`vcpkg.json`](../vcpkg.json) forward-declares the dependency shape.

3. **No `wgpu-native`.** It has no vcpkg port and would require a Rust toolchain in CI. ADR-0014 keeps it as a documented opt-in fallback for environments where Dawn is fussy (some Linux mesa stacks).

4. **xeus-cpp is the notebook kernel** (replacing the frozen xeus-cling). The kernel name (`xcpp20`) is the same, so this notebook works on either kernel — but only xeus-cpp handles the class-type NTTPs that `TypedTensor` uses, so any future cell demonstrating compile-time-labelled tensors will require xeus-cpp.

## 5. Performance expectations (honest framing)

Per [ADR-0001](../docs/arc42/09-decisions/0001-pivot-to-educational-named-axis-dsl.md) and the canonical-reference framing, performance is a quality-4 concern that **must not block** quality 1–3 (clarity, correctness, portability). The WebGPU adapter exists primarily as the architectural payoff of the Hexagonal-lite design — it is *possible* without changing the Domain.

When the dispatch wiring lands (P3.M3.2 / P3.M4.2) and runs on a real GPU, the expected envelopes are:

| Operation | Reference baseline | Eigen | WebGPU | Why |
| --------- | ------------------ | ----- | ------ | --- |
| `add(a, b)`, 1M elements | ~1 ms | ~0.5 ms | ~20 ms | Host↔device transfer (~10 ms each way) dominates compute (<0.1 ms). |
| matvec 1024 × 1024 | ~50 ms | ~5 ms | ~5 ms (with matvec inefficiency note) | Compute ≈ transfer; 1-D effective dispatch wastes 15 / 16 threads. |
| matmul 512 × 512 × 512 | ~574 ms | ~10 ms | ~10 ms | Compute amortises transfer; the canonical Phase 3 pitch. |

**Conclusion**: WebGPU is competitive with optimised CPU BLAS for compute-bound ops at moderate-to-large sizes. For small element-wise ops, CPU wins on round-trip alone. The canonical learner-facing takeaway, from tutorial 06's design framing, is *not* that WebGPU is universally faster — it is that **the same Domain code reaches the GPU when the workload warrants it**, with one CMake variable's worth of opt-in cost.

## 6. What unblocks live execution

When the maintainer provisions:

1. **A self-hosted GitHub Actions runner with a Dawn-compatible GPU.** Per [ADR-0012 §Decision Outcome point 6](../docs/arc42/09-decisions/0012-webgpu-adapter-implementation-design.md), the project's CI policy is compile-only on generic runners + numerical-agreement on a self-hosted runner.

2. **A vcpkg baseline bump to a revision including `dawn@20260410.140140`+** (e.g. tag `2026.04.27`, commit `56bb2411609227288b70117ead2c47585ba07713`). The same bump pulls Eigen 3.4 → 5.0.1; the P3.M3.2 PR bundles them with focused CI verification.

…then P3.M3.2 and P3.M4.2 land the dispatch wiring per §3 above. This notebook gains executable cells exercising the kernels live. The cross-validation suite in [`tests/test_webgpu_backend.cpp`](../tests/test_webgpu_backend.cpp) flips from stub-pass to real-GPU-pass with a tolerance softened from `1e-9` to `1e-5` (for `float`, per ADR-0012).

## 7. Cross-references

**Detailed designs (line-resolved):**
- [`docs/detailed-design/webgpu-element-wise-kernels.md`](../docs/detailed-design/webgpu-element-wise-kernels.md) — P3.M3 element-wise design.
- [`docs/detailed-design/webgpu-gemm-kernel.md`](../docs/detailed-design/webgpu-gemm-kernel.md) — P3.M4 GEMM design.

**ADRs anchoring this notebook:**
- [ADR-0006](../docs/arc42/09-decisions/0006-adopt-webgpu-as-gpu-backend.md) — WebGPU as the GPU backend choice.
- [ADR-0011](../docs/arc42/09-decisions/0011-kernel-backend-port-api.md) — `KernelBackend` port API.
- [ADR-0012](../docs/arc42/09-decisions/0012-webgpu-adapter-implementation-design.md) — WebGPU adapter implementation design.
- [ADR-0014](../docs/arc42/09-decisions/0014-external-substrate-strategy.md) — external substrate strategy.

**Reports & plans:**
- [Phase 3 impl-plan](../docs/impl-plans/2026-05-11_phase-3-webgpu.md) — full milestone list.
- [External substrate research (2026-05-11)](../docs/reports/2026-05-11_external-substrate-research.md) — why gpu.cpp / Dawn / xeus-cpp / stdBLAS.
- [Phase 4 release rehearsal (2026-05-11)](../docs/reports/2026-05-11_phase-4-release-rehearsal.md) — recommends Option 3 design-walkthrough mode for tutorial 06 (i.e. this notebook).

**Vendored source:**
- [`third_party/gpu_cpp/gpu.hpp`](../third_party/gpu_cpp/gpu.hpp) — gpu.cpp at tag 0.2.0 (Apache-2.0).
- [`include/tensor/core/backend/webgpu_wgsl.hpp`](../include/tensor/core/backend/webgpu_wgsl.hpp) — the eight kernel sources walked through above.
- [`include/tensor/core/backend/webgpu.hpp`](../include/tensor/core/backend/webgpu.hpp) — the stub Backend currently delegating to reference.

**Sibling notebooks:**
- [`tutorials/00_intro.ipynb`](./00_intro.ipynb) — named-axis fundamentals.
- [`tutorials/05_autograd-from-scratch.ipynb`](./05_autograd-from-scratch.ipynb) — autograd primitive-by-primitive.
- [`tutorials/07_mlp-on-toy.ipynb`](./07_mlp-on-toy.ipynb) — capstone training loop.
- [`tutorials/08_swappable-backends.ipynb`](./08_swappable-backends.ipynb) — Hexagonal-lite payoff with reference + Eigen.